In [1]:
# Carga del modelo
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
# Cargamos los datos del faq para poder probar su vectorización
from ingest import load_faq_data
documents = load_faq_data();

In [3]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text);

In [4]:
# Vectorización
from tqdm.auto import tqdm
import numpy as np
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

X = np.array(vectors)

X.shape

  0%|          | 0/27 [00:00<?, ?it/s]

(1350, 384)

In [5]:
# Guardamos la vectorización en un índice.
# Este índice asocia cada vector a un documento en base al índice del array.
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields = ["course"])
vindex.fit(X, documents);

In [8]:
# Búsqueda de ejemplo, similar a la del tema 1 al compartir API.
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
  'doc_id': '41aabbd7c5'},
 {'course': 'mlops-zoomcamp',
  'section': 'General Cour

In [11]:
# Misma búsqueda pero filtrando por curso.
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)


In [10]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'doc_id': '69d122f12e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offere

In [12]:
# Creación del rag completo
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [13]:
from ingest import build_index
index = build_index(documents)


In [16]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query);
        filter_dict = {"course": "llm-zoomcamp"}

        return self.index.search(query_vector, num_results = num_results, filter_dict = filter_dict)

In [20]:
from rag_helper import RAGBase

vector_assistant = RAGVector(
    index=vindex,
    embedder = model,
    llm_client=openai_client,
)

In [21]:
query = "I just found out about the program, can I still sign up?"
vector_assistant.rag(query)

'Yes — you can still join. If you want a certificate, make sure to submit your project while submissions are still open.'

In [22]:
vector_assistant.rag("the program has already begun, can I still sign up?")


'Yes, you can still join. You don’t need special approval, and you can start learning and submitting homework while the form is open. If you want a certificate, make sure you submit your project before submissions close.'